In [1]:
import json
import os
try:
    from backend.utils import load_json
except ModuleNotFoundError:
    print("Updating working directory....")
    os.chdir("../")
    # print(os.getcwd())
    from backend.utils import load_json
    
from tqdm import tqdm 
from backend.utils import load_response_dict
from backend.dataset.utils import organize_dataset
from backend.nli_labels import ExNLILabel
import math

import matplotlib.pyplot as plt
import numpy as np

from backend.uncertainty.utils import remove_invalid, calculate_probs, find_relevant_logprobs, sort_correct_wrong
from backend.uncertainty.metrics import calculate_uncertainty
import itertools

Updating working directory....


In [2]:
# load dataset
from pathlib import Path


dataset = load_json("data/contract-nli/test.json")
dataset = organize_dataset(dataset)

output_dir = Path("out/responses/logprobs")
model_dirs = [p for p in output_dir.iterdir() if p.is_dir()]
print(model_dirs)

def match_substr(substr, exclude=None):
    return [p for p in model_dirs if (substr in str(p) and (exclude is None or exclude not in str(p)))]

gemma4 = match_substr("gemma4")
qwen35 = match_substr("qwen3.5")
qwen36 = match_substr("qwen3.6", exclude="nothink")
ministral3 = match_substr("ministral-3")

by_family = [gemma4, qwen35, qwen36, ministral3]
print(by_family)

all_fams = gemma4 + qwen35 + qwen36 + ministral3

response_dirs = {}
for md in all_fams:
    model = md.name
    model_dir = (md) / "narendra_nli_nonbinary_concise"
    print(list(model_dir.iterdir()))
    # point3 = [p for p in model_dir.iterdir() if (p.is_dir() and "0.3" in str(p))][0]
    seed_dirs = [str(p) for p in model_dir.iterdir() if (p.is_dir() and "seed" in str(p))]
    response_dirs[model] = (seed_dirs, "classification")
# print(response_dirs)

[PosixPath('out/responses/logprobs/ministral-3_14b'), PosixPath('out/responses/logprobs/gemma4_e2b'), PosixPath('out/responses/logprobs/ministral-3_8b'), PosixPath('out/responses/logprobs/gemma4_26b'), PosixPath('out/responses/logprobs/ministral-3_3b'), PosixPath('out/responses/logprobs/qwen3.5_4b'), PosixPath('out/responses/logprobs/gemma4_e4b'), PosixPath('out/responses/logprobs/qwen3.6_35b'), PosixPath('out/responses/logprobs/qwen3.5_4b_think'), PosixPath('out/responses/logprobs/qwen3.5_27b'), PosixPath('out/responses/logprobs/qwen3.5_9b')]
[[PosixPath('out/responses/logprobs/gemma4_e2b'), PosixPath('out/responses/logprobs/gemma4_26b'), PosixPath('out/responses/logprobs/gemma4_e4b')], [PosixPath('out/responses/logprobs/qwen3.5_4b'), PosixPath('out/responses/logprobs/qwen3.5_4b_think'), PosixPath('out/responses/logprobs/qwen3.5_27b'), PosixPath('out/responses/logprobs/qwen3.5_9b')], [PosixPath('out/responses/logprobs/qwen3.6_35b')], [PosixPath('out/responses/logprobs/ministral-3_14b'

In [3]:


    # load and process responses

model_responses = {}
by_seed = {}
for model, (response_dir, structure) in response_dirs.items():
    by_seed[model] = {}
    correct, wrong = {}, {}
    for fp in response_dir:
        try:
            responses = load_response_dict(fp)
        except FileNotFoundError:
            print("File not found")
            continue
        remove_invalid(responses)

        # add probabilities to responses
        # calculate_probs(responses)
        print(fp)
        find_relevant_logprobs(responses, relevant_scope=None, structure=structure)
        calculate_uncertainty(responses)

        # find correct/wrong
        c, w = sort_correct_wrong(dataset, responses)

        # complicated indexing to save on boilerplating 
        for big_dict, sub_dict in [(correct, c), (wrong, w)]:
            if len(big_dict) == 0:
                for k,v in sub_dict.items():
                    big_dict[k] = v
            else:
                for k,v in big_dict.items():
                    big_dict[k] = v + sub_dict[k]
        
        seed = int(fp.split("/")[-1][4:])
        by_seed[model][seed] = {
            "correct": c,
            "wrong": w
        }
    
    model_responses[model] = {
        "correct": correct,
        "wrong": wrong
    }

Loading responses:   0%|          | 0/123 [00:00<?, ?it/s]

Loading responses: 100%|██████████| 123/123 [00:02<00:00, 60.00it/s]


Removed 1 invalid answers!
out/responses/logprobs/gemma4_e2b/narendra_nli_nonbinary_concise/seed0


Sorting: 100%|██████████| 123/123 [00:00<00:00, 34476.04it/s]


Num correct: 1461
Num wrong:   629


...

Num correct: 116
Num wrong:   94


Loading responses: 100%|██████████| 123/123 [00:00<00:00, 700.92it/s]


Removed 0 invalid answers!
out/responses/logprobs/ministral-3_3b/narendra_nli_nonbinary_concise/seed42


Sorting: 100%|██████████| 123/123 [00:00<00:00, 79614.10it/s]

Num correct: 115
Num wrong:   95


In [135]:
by_seed["gemma4_e2b"][0]["correct"]["NotMentioned"][0]["uncertainty"]

{'entropy': np.float64(0.7322386304826939),
 'average_logprob': np.float64(0.5579672302029999),
 'perplexity': np.float64(1.7471174006868007),
 'maximum_logprob': np.float64(21.30946159362793),
 'response_improbability': np.float64(1.0)}

In [4]:
def set_boxplot_line_color(bp, color, width=2.):
    for element in ['boxes', 'whiskers', 'caps', 'medians']:
        for item in bp[element]:
            item.set_color(color)
            item.set_linewidth(width)

def big_boxplot(correct: dict, wrong: dict, save_file: str, metric: str, title: str = ""):

    classes = sorted(list(correct.keys()))

    correct_data = []
    wrong_data = []
    for k in classes:
        correct_data.append(correct[k])
        wrong_data.append(wrong[k])

    num_groups = len(classes)
    positions = np.arange(num_groups)

    offset = 0.15
    width = 0.2
    
    fig = plt.figure(figsize=(10,7))
    ax = plt.gca()

    
    bp_correct = plt.boxplot(
        correct_data,
        positions=positions - offset,
        widths=width,
        patch_artist=False,
        showfliers=False
    )

    bp_wrong = plt.boxplot(
        wrong_data,
        positions=positions + offset,
        widths=width,
        patch_artist=False,
        showfliers=False
    )

    # Color the boxes
    set_boxplot_line_color(bp_correct, "blue")
    set_boxplot_line_color(bp_wrong, "red")

    # Axis formatting
    plt.xticks(positions, classes)
    plt.ylabel(metric)
    plt.legend(
        [bp_correct['boxes'][0], bp_wrong['boxes'][0]],
        ['Correct', 'Wrong']
    )
    
    ax.set_xlabel("Response prediction")
    ax.set_title(title)

    return ax, fig

    
# pick model
sorted_model_responses = {}
for k,v in model_responses.items():
    sorted_model_responses[k] = {}

    model_dict = model_responses[k]

    # sort correct/wrong properly, by class
    correct = {}
    wrong = {}
    for cw_dict, cw_str in [[correct, "correct"], [wrong, "wrong"]]:
        ld = model_dict[cw_str]
        for label in ld.keys():
            if label not in cw_dict:
                cw_dict[label] = []
            for el in ld[label]:
                cw_dict[label] += [el["uncertainty"]["perplexity"]]

        sorted_model_responses[k][cw_str] = cw_dict
    
# # plot graph
# ax, fig = boxplot_analysis(correct, wrong, metric="Perplexity", title=f"Perplexity for GPT-OSS 20B", save_file=None)
# fig.show()

In [137]:
sorted_model_responses["gemma4_e2b"]["correct"]["Entailment"][0]

np.float64(1.8494983783330445)

In [5]:
def big_boxplot(correct: dict, wrong: dict, save_file: str, metric: str, title: str = ""):

    classes = sorted(list(correct.keys()))

    correct_data = []
    wrong_data = []
    for k in classes:
        correct_data.append(correct[k])
        wrong_data.append(wrong[k])

    num_groups = len(classes)
    positions = np.arange(num_groups)

    offset = 0.15
    width = 0.2
    
    fig = plt.figure(figsize=(10,7))
    ax = plt.gca()

    
    bp_correct = plt.boxplot(
        correct_data,
        positions=positions - offset,
        widths=width,
        patch_artist=False,
        showfliers=False
    )

    bp_wrong = plt.boxplot(
        wrong_data,
        positions=positions + offset,
        widths=width,
        patch_artist=False,
        showfliers=False
    )

    # Color the boxes
    set_boxplot_line_color(bp_correct, "blue")
    set_boxplot_line_color(bp_wrong, "red")

    # Axis formatting
    plt.xticks(positions, classes)
    plt.ylabel(metric)
    plt.legend(
        [bp_correct['boxes'][0], bp_wrong['boxes'][0]],
        ['Correct', 'Wrong']
    )
    
    ax.set_xlabel("Response prediction")
    ax.set_title(title)

    return ax, fig




In [6]:

def plot_model_grouped_boxplots(data):
    """
    Parameters
    ----------
    data : dict
        {
            "model1": {
                "correct": {"label1": [...], "label2": [...], "label3": [...]},
                "wrong":   {"label1": [...], "label2": [...], "label3": [...]}
            },
            "model2": { ... }
        }
    """

    models = list(data.keys())
    labels = list(next(iter(data.values()))['correct'].keys())

    num_labels = len(labels)
    box_width = 0.3
    pair_offset = 0.2
    model_gap = 1.0

    fig, ax = plt.subplots(figsize=(20,7))

    x_positions = []
    x_labels = []
    model_centers = []

    current_x = 0

    for model in models:
        model_start = current_x

        for label in labels:
            correct_vals = data[model]['correct'][label]
            wrong_vals = data[model]['wrong'][label]

            pos_correct = current_x - pair_offset
            pos_wrong = current_x + pair_offset

            bp_correct = ax.boxplot(
                correct_vals,
                positions=[pos_correct],
                widths=box_width,
                patch_artist=False,
                showfliers=False
            )
            bp_wrong = ax.boxplot(
                wrong_vals,
                positions=[pos_wrong],
                widths=box_width,
                patch_artist=False,
                showfliers=False
            )

            set_boxplot_line_color(bp_correct, 'blue')
            set_boxplot_line_color(bp_wrong, 'red')

            x_positions.append(current_x)
            x_labels.append(label)

            current_x += 1

        model_end = current_x - 1
        model_centers.append((model_start + model_end) / 2)
        current_x += model_gap

    # Label ticks for individual label pairs
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels)

    # Model labels below x-axis
    for center, model in zip(model_centers, models):
        ax.text(
            center,
            -0.05,
            model,
            ha='center',
            va='top',
            transform=ax.get_xaxis_transform(),
            fontsize=10,
            fontweight='bold'
        )

    ax.set_ylabel("Value")

    # Legend
    ax.plot([], [], color='blue', label='correct')
    ax.plot([], [], color='red', label='wrong')
    ax.legend()

    plt.tight_layout()
    plt.show()

    return fig, ax

In [124]:
import copy

In [152]:
def plot_seed_boxplots(data, figsize=(15,7)):
    """
    Parameters
    ----------
    data : dict
        {
            "model1": {
                "correct": {"label1": [...], "label2": [...], "label3": [...]},
                "wrong":   {"label1": [...], "label2": [...], "label3": [...]}
            },
            "model2": { ... }
        }
    """

    seeds = list(data.keys())
    print(data)
    labels = sorted(list(next(iter(data.values()))['correct'].keys()))

    num_labels = len(labels)
    box_width = 0.15
    pair_offset = [0.2, 0.4, 0.6]
    seed_gap = 1.0

    fig, ax = plt.subplots(figsize=figsize)

    x_positions = []
    x_labels = []
    seed_centers = []

    current_x = 0

    for label in labels:
        
        model_start = current_x
        
        correct_vals = []
        wrong_vals = []
        for seed in seeds:
            correct_vals.append(data[seed]['correct'][label])
            wrong_vals.append(data[seed]['wrong'][label])

        pos_correct = [current_x - po for po in pair_offset]
        pos_wrong = [current_x + po for po in pair_offset]
        x_positions.append(current_x)
        x_labels.append(label)
        current_x += 1

        model_end = current_x - 1
        seed_centers.append((model_start + model_end) / 2)
        current_x += seed_gap


        bp_correct = ax.boxplot(
            correct_vals,
            positions=pos_correct,
            widths=box_width,
            patch_artist=False,
            showfliers=False,
        )
        bp_wrong = ax.boxplot(
            wrong_vals,
            positions=pos_wrong,
            widths=box_width,
            patch_artist=False,
            showfliers=False
        )

        set_boxplot_line_color(bp_correct, 'blue', 3.)
        set_boxplot_line_color(bp_wrong, 'red', 3.)


    ax.set_xticks(x_positions)
    ax.set_xticklabels(labels)

    # Legend
    ax.plot([], [], color='blue', label='correct')
    ax.plot([], [], color='red', label='wrong')
    ax.legend()

    return fig, ax

by_metric = {}
for metric in ["maximum_logprob", "average_logprob"]:
    by_metric[metric] = {}
    # ylabels = {
    #     "maximum_logprob": "Maximum log probability", 
    #     "perplexity": "Perplexity",
    #     "average_logprob": "Average log probability"
    # }
    # ylabel = ylabels[metric]
    sorted_seed_responses = {}
    for model in by_seed.keys():
        for seed,seed_dict in by_seed[model].items():
            sorted_seed_responses[seed] = {}


            # sort correct/wrong properly, by class
            correct = {}
            wrong = {}
            for cw_dict, cw_str in [[correct, "correct"], [wrong, "wrong"]]:
                # build cw dict
                ld = seed_dict[cw_str]
                for label in ld.keys():
                    if label not in cw_dict:
                        cw_dict[label] = []
                    for el in ld[label]:
                        cw_dict[label] += [el["uncertainty"][metric]]

                # save cw dict
                sorted_seed_responses[seed][cw_str] = cw_dict
        print(metric)
        by_metric[metric][model] = copy.deepcopy(sorted_seed_responses)
        # fig, ax = plot_seed_boxplots(sorted_seed_responses)
        # plt.rcParams.update({'font.size': 23})
        # ax.set_ylim(bottom=0)
        # ax.set_ylabel(ylabel, weight="bold")
        # ax.legend(loc="lower right")
        # ax.set_xlabel(f"{model} prediction", weight="bold")
        # fig.tight_layout()
        # # fig.show()
        
        # os.makedirs("out/figs/resubmission/by_seed/", exist_ok=True)
        # save_name = f"out/figs/resubmission/by_seed/{model}_{metric}_by_seed.pdf"
        # fig.savefig(save_name)

maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
maximum_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob
average_logprob


In [153]:
from scipy.stats import mannwhitneyu
from tabulate import tabulate


table = []

pval_dict = dict([(model, {}) for model in list(by_metric.values())[0].keys()])
for model, model_dict in pval_dict.items():
    pval_dict[model] = {
        "Entailment": {},
        "Contradiction": {},
        "NotMentioned": {}
    }
    for label, label_dict in pval_dict[model].items():
        pval_dict[model][label] = dict((metric, {}) for metric in sorted(list(by_metric.keys())))
# print(pval_dict)



for metric, metric_dict in by_metric.items():
    for model, model_dict in metric_dict.items():
        for seed, seed_dict in model_dict.items():
            labels = list(seed_dict["correct"].keys())
            for label in labels:
                correct_dist = seed_dict["correct"][label]
                wrong_dist = seed_dict["wrong"][label]
                stat, p = mannwhitneyu(correct_dist, wrong_dist, alternative="less")
                # print(f"{metric}-{model}-{label}-{seed}:", stat, p)
                table.append([metric, model, label, seed, stat, p])
                pval_dict[model][label][metric][seed] = round(p, 3)


# \multirow{3}{*}{Model A}
#     & Entailment & val & val & val & val & val & val & val & val & val  \\
#     & Contradiction& val & val & val & val & val & val & val & val & val  \\
#     & NotMentioned & val & val & val & val & val & val & val & val & val  \\
with open("mannwhitney-less.txt", "w") as f:
    for model, model_dict in pval_dict.items():
        f.write("\multirow{3}{*}{" + model + "}\n")
        for label, label_dict in model_dict.items():
            f.write(f"\t & {label}")
            for metric, metric_dict in label_dict.items():
                vals = ""
                print(metric, end=" ")
                for seed in [0, 1, 42]:
                    vals += " & "
                    p = metric_dict[seed]
                    print(p, end=" ")
                    if p < 0.05:
                        vals += "\cellcolor{SpringGreen} "

                    if p < 0.01:
                        vals += " p < 0.01"
                    else:
                        vals += f" {p}"
                f.write(vals)
            print("")
            f.write(" \\\\\n")
        f.write("\hline\n")
print(tabulate(table, headers=["Metric", "Model", "Label", "Seed", "Stat", "P"]))

average_logprob 1.0 1.0 1.0 maximum_logprob 1.0 1.0 1.0 
average_logprob 0.14 0.123 0.232 maximum_logprob 1.0 0.998 0.997 
average_logprob 0.712 0.763 0.968 maximum_logprob 0.0 0.0 0.0 
average_logprob 1.0 1.0 1.0 maximum_logprob 1.0 1.0 1.0 
average_logprob 0.14 0.123 0.232 maximum_logprob 1.0 0.998 0.997 
average_logprob 0.712 0.763 0.968 maximum_logprob 0.0 0.0 0.0 
average_logprob 0.009 0.002 0.006 maximum_logprob 0.001 0.0 0.0 
average_logprob 0.0 0.0 0.0 maximum_logprob 0.0 0.0 0.0 
average_logprob 1.0 0.999 0.999 maximum_logprob 0.543 0.228 0.468 
average_logprob 0.688 0.417 0.235 maximum_logprob 0.771 0.519 0.29 
average_logprob 0.083 0.587 0.19 maximum_logprob 0.266 0.453 0.213 
average_logprob 0.674 0.566 0.973 maximum_logprob 0.598 0.434 0.701 
average_logprob 0.484 0.681 0.73 maximum_logprob 0.155 0.451 0.303 
average_logprob 0.731 0.618 0.34 maximum_logprob 0.988 0.946 0.98 
average_logprob 0.988 0.936 0.957 maximum_logprob 0.12 0.452 0.496 
average_logprob 0.241 0.681 0.0